<a href="https://colab.research.google.com/github/bilgeerol/bim448-sentiment/blob/main/notebooks/02_vader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 – VADER
**Sahibi:** Zehra · **Amaç:** VADER ile İngilizce test setini ve Türkçe test setini (doğrudan ve İngilizceye çeviri sonrası) sınıflandırmak.
**Girdi:** `data/processed/en_test.csv`, `tr_test.csv`, `tr_test_translated.csv`
**Çıktı:** `predictions/vader_en.csv`, `vader_tr_direct.csv`, `vader_tr_mt.csv`, `data/processed/tr_test_translated.csv`

In [3]:
# 0. KURULUM HÜCRESİ — her oturumda ilk bu çalıştırılır
from google.colab import userdata, drive
import os

drive.mount('/content/drive')

TOKEN = userdata.get('GITHUB_TOKEN')
REPO_OWNER = "bilgeerol"
REPO = "bim448-sentiment"
REPO_DIR = f"/content/{REPO}"

if not os.path.exists(REPO_DIR):
    !git clone https://{TOKEN}@github.com/{REPO_OWNER}/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git pull
!git config user.name "zehranurBurmaoglu"
!git config user.email "znburmaoglu@gmail.com"

# Kontrol: depo geldi mi, push iznim var mı?
!ls
!git push --dry-run

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/bim448-sentiment
Already up to date.
data  notebooks  predictions  README.md  report  requirements.txt  results  src
Everything up-to-date


## 1. VADER İngilizce

In [8]:
# 1.1 VADER'ı kur, sürümünü kaydet ve birkaç örnekle kuralların etkisini gör
!pip install -q vaderSentiment
import importlib.metadata
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

print("vaderSentiment sürümü:", importlib.metadata.version("vaderSentiment"))
analyzer = SentimentIntensityAnalyzer()

ornekler = [
    "The product is good.",
    "The product is GOOD!!!",
    "The product is not good.",
    "The product is good but very expensive.",
    "I received the product yesterday.",
    "Ürün çok güzel, herkese tavsiye ederim.",
]
for cumle in ornekler:
    print(f"{analyzer.polarity_scores(cumle)}  ←  {cumle}")

vaderSentiment sürümü: 3.3.2
{'neg': 0.0, 'neu': 0.508, 'pos': 0.492, 'compound': 0.4404}  ←  The product is good.
{'neg': 0.0, 'neu': 0.4, 'pos': 0.6, 'compound': 0.6714}  ←  The product is GOOD!!!
{'neg': 0.376, 'neu': 0.624, 'pos': 0.0, 'compound': -0.3412}  ←  The product is not good.
{'neg': 0.0, 'neu': 0.755, 'pos': 0.245, 'compound': 0.2382}  ←  The product is good but very expensive.
{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}  ←  I received the product yesterday.
{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}  ←  Ürün çok güzel, herkese tavsiye ederim.


In [9]:
# 1.2 İngilizce test setini VADER ile sınıflandır ve predictions/vader_en.csv olarak kaydet
import pandas as pd, time

def vader_etiket(compound):
    """VADER'ın standart eşikleri: ≥ 0,05 positive, ≤ −0,05 negative, arası neutral."""
    if compound >= 0.05:
        return "positive"
    if compound <= -0.05:
        return "negative"
    return "neutral"

en_test = pd.read_csv("data/processed/en_test.csv", encoding="utf-8")

bas = time.time()
skorlar = en_test["text"].apply(lambda t: analyzer.polarity_scores(t)["compound"])
sure = time.time() - bas

vader_en = pd.DataFrame({
    "id": en_test["id"],
    "text": en_test["text"],
    "true_label": en_test["label"],
    "pred_label": skorlar.apply(vader_etiket),
    "score": skorlar,
})
vader_en.to_csv("predictions/vader_en.csv", index=False, encoding="utf-8")

# Kontroller: dosyayı diskten tekrar okuyup ortak kurallara uyuyor mu bak
kontrol = pd.read_csv("predictions/vader_en.csv", encoding="utf-8")
print("Satır:", len(kontrol), "| sütunlar:", list(kontrol.columns))
print("id'ler ve sıra test setiyle aynı:", (kontrol["id"].values == en_test["id"].values).all())
print("Geçersiz tahmin etiketi:", (~kontrol["pred_label"].isin(["negative", "neutral", "positive"])).sum())
print(f"Skorlama süresi: {sure:.1f} sn")
print("\nTahmin dağılımı:")
print(kontrol["pred_label"].value_counts().reindex(["negative", "neutral", "positive"]))
print("\ncompound = 0 olan metin sayısı:", (kontrol["score"] == 0).sum())

Satır: 3000 | sütunlar: ['id', 'text', 'true_label', 'pred_label', 'score']
id'ler ve sıra test setiyle aynı: True
Geçersiz tahmin etiketi: 0
Skorlama süresi: 2.1 sn

Tahmin dağılımı:
pred_label
negative     741
neutral      380
positive    1879
Name: count, dtype: int64

compound = 0 olan metin sayısı: 334


In [ ]:
# 1.3 VADER İngilizce tahmin dosyasını GitHub'a gönder
!git add predictions/vader_en.csv
!git commit -m "[Zehra] VADER İngilizce tahminleri eklendi"
!git pull --rebase
!git push

[main 9a405b2] [Zehra] VADER İngilizce tahminleri eklendi
 1 file changed, 3001 insertions(+)
 create mode 100644 predictions/vader_en.csv
Current branch main is up to date.
Enumerating objects: 6, done.
Counting objects: 100% (6/6), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 221.94 KiB | 4.53 MiB/s, done.
Total 4 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/bilgeerol/bim448-sentiment.git
   4b15326..9a405b2  main -> main


## 2. Türkçe→İngilizce çeviri

In [4]:
# 2.1 GPU'yu doğrula, çeviri modelini yükle ve 20 metinlik deneme çevirisiyle süreyi ölç
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q transformers sentencepiece sacremoses
import torch, time, importlib.metadata
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("GPU kullanılabilir mi:", torch.cuda.is_available())
print("transformers sürümü:", importlib.metadata.version("transformers"))

MT_MODEL = "Helsinki-NLP/opus-mt-tc-big-tr-en"
cihaz = "cuda" if torch.cuda.is_available() else "cpu"
mt_tokenizer = AutoTokenizer.from_pretrained(MT_MODEL)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL).to(cihaz).eval()

def cevir(metinler, batch_size=32):
    """Metin listesini batch'ler halinde Türkçeden İngilizceye çevirir."""
    sonuc = []
    for i in range(0, len(metinler), batch_size):
        parca = metinler[i:i + batch_size]
        girdi = mt_tokenizer(parca, return_tensors="pt", padding=True,
                             truncation=True, max_length=512).to(cihaz)
        with torch.no_grad():
            cikti = mt_model.generate(**girdi, max_new_tokens=512)
        sonuc.extend(mt_tokenizer.batch_decode(cikti, skip_special_tokens=True))
    return sonuc

tr_test = pd.read_csv("data/processed/tr_test.csv", encoding="utf-8")
deneme = tr_test["text"].head(20).tolist()

bas = time.time()
deneme_en = cevir(deneme)
sure = time.time() - bas
print(f"\n20 metin: {sure:.1f} sn → 3.000 metin için kaba tahmin: {sure / 20 * 3000 / 60:.1f} dk\n")

for tr, en in list(zip(deneme, deneme_en))[:5]:
    print("TR:", tr)
    print("EN:", en, "\n")

name, memory.total [MiB]
Tesla T4, 15360 MiB
GPU kullanılabilir mi: True
transformers sürümü: 5.22.1


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=512) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



20 metin: 2.9 sn → 3.000 metin için kaba tahmin: 7.2 dk

TR: Yorumlar hep olumlu yönde ama ben ne yazıkki sevemedim topak topak kalıyor dudakta.
EN: The comments are always positive, but unfortunately I didn't like it, the lump stays on the lip. 

TR: Özenle kargolanmış kullanınca yazacağım
EN: I'll write it down when I use it carefully. 

TR: tavsiye üzerine almıştım ama kesinlikle çatlak oluşumu genetiğinize bağlı. faydasını göremedim.
EN: I took it on advice, but it certainly depends on your fracture formation genetics. I didn't see the benefit. 

TR: Saklama kutusu yoktu, taşları da kaliteli değil
EN: There was no storage box, and the stones were not of good quality. 

TR: kesinlikle tavsiye ederim çünkü yüze masaj olarak uygulandığı için kan dolaşımını hızlandırıyor ve yüzü canlandırıyor. aynı zamanda anti-aging etkisi var.
EN: I would definitely recommend it because it is applied as a massage to the face, which speeds up blood circulation and stimulates the face. It also has an 

In [5]:
# 2.2 3.000 Türkçe yorumu çevir, data/processed/tr_test_translated.csv olarak kaydet ve kontrol et
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()          # zararsız uzunluk uyarısının her batch'te tekrarlanmasını engelle

bas = time.time()
ceviriler = cevir(tr_test["text"].tolist(), batch_size=32)
ceviri_suresi = time.time() - bas

tr_ceviri = pd.DataFrame({
    "id": tr_test["id"],
    "text": tr_test["text"],          # Türkçe orijinal
    "text_en": ceviriler,             # İngilizce çeviri
    "label": tr_test["label"],
})
tr_ceviri.to_csv("data/processed/tr_test_translated.csv", index=False, encoding="utf-8")

# Kontroller
k = pd.read_csv("data/processed/tr_test_translated.csv", encoding="utf-8", keep_default_na=False)
print("Satır:", len(k), "| sütunlar:", list(k.columns))
print("id sırası test setiyle aynı:", (k["id"].values == tr_test["id"].values).all())
print("Boş çeviri sayısı:", (k["text_en"].str.strip() == "").sum())
print("512 tokeni aşan (kesilen) Türkçe metin:",
      sum(len(mt_tokenizer(t)["input_ids"]) > 512 for t in tr_test["text"]))
print(f"Toplam çeviri süresi: {ceviri_suresi / 60:.1f} dk")

# Gözle kontrol için 10 rastgele çeviri (rapor için anlamı bozulanları not alacağız)
for _, r in k.sample(10, random_state=1).iterrows():
    print(f"\n[{r['id']}] ({r['label']})\nTR: {r['text']}\nEN: {r['text_en']}")

Satır: 3000 | sütunlar: ['id', 'text', 'text_en', 'label']
id sırası test setiyle aynı: True
Boş çeviri sayısı: 0
512 tokeni aşan (kesilen) Türkçe metin: 0
Toplam çeviri süresi: 3.1 dk

[tr_106630] (positive)
TR: 2 aydır kullanıyorum. Şarjı hiç bitmedi daha, yüzümü çok güzel temizliyor cildimde gözle görülür düzelme oldu. Çok memnunum.
EN: I've been using it for 2 months, and it's not finished yet, it cleans my face very well, it's a noticeable improvement in my skin. I'm very happy.

[tr_133158] (neutral)
TR: ürünü bitirmek uzereyim geçen ay almıştım sanki mimik çizgilerim doldu ama sürekli kullanmak gerekli
EN: I'm going to finish the product last month as if I've had my mimic lines filled but need to use them all the time.

[tr_012981] (negative)
TR: Gayet güzel fiyatı 58 olunca aldım
EN: I got it when the price was 58, which is pretty good.

[tr_085488] (neutral)
TR: Pakette sızma vardı. Ama şampuanı seviyorum. Yumuşak his veriyor saça
EN: There was infiltration in the package, but

In [6]:
# 2.3 Çeviri dosyasını GitHub'a gönder
!git add data/processed/tr_test_translated.csv
!git commit -m "[Zehra] Türkçe test setinin İngilizce çevirisi eklendi (opus-mt-tc-big-tr-en)"
!git pull --rebase
!git push

[main b1c528f] [Zehra] Türkçe test setinin İngilizce çevirisi eklendi (opus-mt-tc-big-tr-en)
 1 file changed, 3001 insertions(+)
 create mode 100644 data/processed/tr_test_translated.csv
Current branch main is up to date.
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (5/5), 236.17 KiB | 4.72 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/bilgeerol/bim448-sentiment.git
   8ca611e..b1c528f  main -> main


## 3. VADER Türkçe (doğrudan ve çeviri sonrası)

In [10]:
# 3.1 VADER Türkçe: doğrudan (Türkçe metin) ve çeviri sonrası (text_en) iki varyant
SINIF = ["negative", "neutral", "positive"]
tr_ceviri = pd.read_csv("data/processed/tr_test_translated.csv", encoding="utf-8", keep_default_na=False)

for varyant, sutun, dosya in [("doğrudan", "text", "predictions/vader_tr_direct.csv"),
                              ("çeviri sonrası", "text_en", "predictions/vader_tr_mt.csv")]:
    bas = time.time()
    skor = tr_ceviri[sutun].apply(lambda t: analyzer.polarity_scores(t)["compound"])
    sure = time.time() - bas

    cikti = pd.DataFrame({"id": tr_ceviri["id"], "text": tr_ceviri["text"],
                          "true_label": tr_ceviri["label"],
                          "pred_label": skor.apply(vader_etiket), "score": skor})
    if sutun == "text_en":                       # çeviri varyantında İngilizce metni de sakla
        cikti.insert(2, "text_en", tr_ceviri["text_en"])
    cikti.to_csv(dosya, index=False, encoding="utf-8")

    k = pd.read_csv(dosya, encoding="utf-8")
    print(f"===== VADER TR {varyant} → {dosya}")
    print("Satır:", len(k), "| sütunlar:", list(k.columns))
    print("id sırası test setiyle aynı:", (k["id"].values == tr_ceviri["id"].values).all(),
          "| geçersiz etiket:", (~k["pred_label"].isin(SINIF)).sum(), f"| süre: {sure:.1f} sn")
    print("Tahmin dağılımı:", k["pred_label"].value_counts().reindex(SINIF).to_dict())
    print(f"neutral tahmin: {(k['pred_label'] == 'neutral').sum()} (%{(k['pred_label'] == 'neutral').mean() * 100:.1f})"
          f" | compound = 0: {(k['score'] == 0).sum()}\n")

# Doğrudan varyantta VADER'ın "bir şey bulduğu" metinlerin kaçında emoji var?
direct = pd.read_csv("predictions/vader_tr_direct.csv", encoding="utf-8")
emoji_var = direct["text"].str.contains(r"[\U0001F300-\U0001FAFF\u2600-\u27BF]", regex=True)
sifir_degil = direct["score"] != 0
print(f"Doğrudan varyant: compound ≠ 0 olan {sifir_degil.sum()} metnin {(sifir_degil & emoji_var).sum()} tanesinde emoji var.")

===== VADER TR doğrudan → predictions/vader_tr_direct.csv
Satır: 3000 | sütunlar: ['id', 'text', 'true_label', 'pred_label', 'score']
id sırası test setiyle aynı: True | geçersiz etiket: 0 | süre: 1.3 sn
Tahmin dağılımı: {'negative': 35, 'neutral': 2811, 'positive': 154}
neutral tahmin: 2811 (%93.7) | compound = 0: 2811

===== VADER TR çeviri sonrası → predictions/vader_tr_mt.csv
Satır: 3000 | sütunlar: ['id', 'text', 'text_en', 'true_label', 'pred_label', 'score']
id sırası test setiyle aynı: True | geçersiz etiket: 0 | süre: 2.0 sn
Tahmin dağılımı: {'negative': 780, 'neutral': 503, 'positive': 1717}
neutral tahmin: 503 (%16.8) | compound = 0: 472

Doğrudan varyant: compound ≠ 0 olan 189 metnin 90 tanesinde emoji var.


In [11]:
# 3.2 İki Türkçe VADER tahmin dosyasını GitHub'a gönder
!git add predictions/vader_tr_direct.csv predictions/vader_tr_mt.csv
!git commit -m "[Zehra] VADER Türkçe tahminleri eklendi (doğrudan ve çeviri sonrası)"
!git pull --rebase
!git push

[main f8f0165] [Zehra] VADER Türkçe tahminleri eklendi (doğrudan ve çeviri sonrası)
 2 files changed, 6002 insertions(+)
 create mode 100644 predictions/vader_tr_direct.csv
 create mode 100644 predictions/vader_tr_mt.csv
Current branch main is up to date.
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 267.57 KiB | 1.69 MiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 2 local objects.
To https://github.com/bilgeerol/bim448-sentiment.git
   b1c528f..f8f0165  main -> main
